# Explorando os arquivos de filmes e usuários

## 1 Filmes

### 1.1 Lendo arquivo de Filmes

In [0]:
# recupera dados de filmes do arquivo para um dataframe
files_path = "/Volumes/academy/dados-streaing-filmes"
df_filmes  = spark.read.csv(f"{files_path}/catalogo_filmes.csv", header=True, inferSchema=True)
print((df_filmes.count(), len(df_filmes.columns)))

(62, 7)


In [0]:
df_filmes.printSchema()

root
 |-- movie_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- duration_minutes: string (nullable = true)
 |-- content_type: string (nullable = true)
 |-- original_language: string (nullable = true)



In [0]:
# verifica como eh o dataframe
df_filmes.limit(5).display()

movie_id,title,genre,release_year,duration_minutes,content_type,original_language
M0001,Cidade Invisível,Animação,2000,163,series,null
M0002,Horizonte 2,Action,1997,76,series,en
M0003,O Último Trem 3,Animação,2002,80,series,EN-US
M0004,Planeta Azul 4,Drama,2002,66,Movie,EN-US
M0005,Mar Aberto 5,Documentary,2019,75,movie,null


In [0]:
# cria tabela temporaria para ser acessada por SQL
df_filmes.createOrReplaceTempView('filmes')

In [0]:
%sql
-- conta quantidade de valores unicos em cada coluna
select 'unicos' as valores
    ,count(distinct movie_id) as movie_id
    ,count(distinct title) as title
    ,count(distinct genre) as genre
    ,count(distinct release_year) as release_year
    ,count(distinct duration_minutes) as duration_minutes
    ,count(distinct content_type) as content_type
    ,count(distinct original_language) as original_language
    ,count(*) as total_linhas
from filmes
union
select 'nulos' as valores
    ,count_if(movie_id is null) as movie_id
    ,count_if(title is null) as title
    ,count_if(genre is null) as genre
    ,count_if(release_year is null) as release_year
    ,count_if(duration_minutes is null) as duration_minutes
    ,count_if(content_type is null) as content_type
    ,count_if(original_language is null) as original_language
    ,count(*) as total_linhas
from filmes

valores,movie_id,title,genre,release_year,duration_minutes,content_type,original_language,total_linhas
unicos,61,54,14,31,50,5,5,62
nulos,0,0,10,3,4,8,12,62


### 1.2 Resolve duplicação em `movie_id`

In [0]:
%sql
-- verifica linha ID de filme duplicado
select a.*
from filmes a 
    inner join (select movie_id from filmes group by movie_id having count(*) > 1) b on a.movie_id = b.movie_id

movie_id,title,genre,release_year,duration_minutes,content_type,original_language
M0005,Mar Aberto 5,Documentary,2019,75,movie,null
M0005,Mar Aberto 5,Documentary,2019,75,movie,null


In [0]:
# remove duplicatas considerando coluna movie_id
df_filmes = df_filmes.dropDuplicates(subset=['movie_id'])
print((df_filmes.count(), len(df_filmes.columns)))

(61, 7)


### 1.3 Padroniza valores de `genre`

In [0]:
%sql
select genre, count(*)
from filmes 
group by genre
order by 1 desc

genre,count(*)
Terror,3
Suspense,5
Sci-Fi,2
Romance,3
Horror,4
Drama,3
Documentário,1
Documentary,7
Comédia,3
Comedy,3


In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit

(df_filmes
 .withColumn("genre", lower(trim(col("genre")))) # padroniza formato
 .groupBy("genre")
  .agg(count(lit(1)).alias("qtd_valores"))
  .sort(df_filmes.genre)).display()

genre,qtd_valores
null,10
action,9
animation,3
animação,4
ação,2
comedy,3
comédia,3
documentary,6
documentário,1
drama,3


In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit, create_map, when
from itertools import chain

map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
map_expr_genre = create_map([lit(x) for x in chain(*map_genre.items())])

(df_filmes
 .withColumn("genre", lower(trim(col("genre")))) # padroniza formato
 .withColumn("genre", when(col("genre").isin(list(map_genre.keys())), map_expr_genre[col("genre")]).otherwise(col("genre"))) # padroniza valores
 .groupBy("genre")
  .agg(count(lit(1)).alias("qtd_valores"))
  .sort(df_filmes.genre)).display()

genre,qtd_valores
null,10
animação,7
ação,11
comédia,6
documentário,7
drama,3
horror,4
romance,3
sci-fi,2
suspense,5


In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit, create_map, when
from itertools import chain

map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
map_expr_genre = create_map([lit(x) for x in chain(*map_genre.items())])

df_filmes = (df_filmes
             .withColumn("genre", lower(trim(col("genre"))))
             .withColumn("genre", when(col("genre").isin(list(map_genre.keys())), map_expr_genre[col("genre")]).otherwise(col("genre"))))

### 1.4 Substitui valores inválidos e converte tipo de `release_year`

In [0]:
%sql
select release_year, count(*)
from filmes 
group by release_year
order by 1 desc

release_year,count(*)
abc,1
2050,1
2026,1
2025,1
2024,2
2023,2
2022,1
2021,2
2020,2
2019,5


In [0]:
from pyspark.sql.functions import col, count, lit, when, expr
from datetime import datetime

current_year = int(datetime.now().year)
# print(current_year)

(df_filmes
 .withColumn('release_year', expr('try_cast(release_year as int)'))
 .withColumn('release_year', when(col("release_year") <= current_year, col("release_year")).otherwise(None))
 .groupBy("release_year")
 .agg(count(lit(1)).alias("qtd_valores"))
 .sort(df_filmes.release_year.desc())
 .display())

release_year,qtd_valores
2026,1
2025,1
2024,2
2023,2
2022,1
2021,2
2020,2
2019,4
2018,2
2016,4


In [0]:
from pyspark.sql.functions import col, count, lit, when, expr
from datetime import datetime

current_year = int(datetime.now().year)
df_filmes    = (df_filmes
                .withColumn('release_year', expr('try_cast(release_year as int)'))
                .withColumn('release_year', when(col("release_year") <= current_year, col("release_year")).otherwise(None)))

### 1.5 Substitui valores inválidos e converte tipo de `duration_movie`

In [0]:
%sql
select duration_minutes, count(*)
from filmes 
group by duration_minutes
order by 1 desc

duration_minutes,count(*)
muito longo,1
99,1
93,1
89,1
87,1
86,2
84,1
82,1
80,1
76,1


In [0]:
from pyspark.sql.functions import col, count, lit, expr

(df_filmes
 .withColumn('duration_minutes', expr('try_cast(duration_minutes as int)'))
 .groupBy("duration_minutes")
 .agg(count(lit(1)).alias("qtd_valores"))
 .sort(df_filmes.duration_minutes.desc())
 .display())

duration_minutes,qtd_valores
178,1
177,1
175,2
174,1
172,1
169,1
165,1
163,1
158,1
154,1


In [0]:
from pyspark.sql.functions import col, count, lit, expr

df_filmes = (df_filmes.withColumn('duration_minutes', expr('try_cast(duration_minutes as int)')))

### 1.6 Padroniza valores de `content_type`

In [0]:
%sql
select content_type, count(*)
from filmes 
group by content_type
order by 1 desc

content_type,count(*)
series,10
movie,13
miniseries,11
SERIES,11
Movie,9
null,8


In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit

(df_filmes
 .withColumn("content_type", lower(trim(col("content_type"))))
 .groupBy("content_type")
  .agg(count(lit(1)).alias("qtd_valores"))
  .sort(df_filmes.content_type)).display()

content_type,qtd_valores
null,8
miniseries,11
movie,21
series,21


In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit

df_filmes = (df_filmes.withColumn("content_type", lower(trim(col("content_type")))))

### 1.7 Padroniza valores de `original_language`

In [0]:
%sql
select original_language, count(*)
from filmes 
group by original_language
order by 1 desc

original_language,count(*)
pt-BR,11
pt,10
es,10
en,6
EN-US,13
null,12


In [0]:
from pyspark.sql.functions import col, count, lit, create_map, when
from itertools import chain

map_lang      = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
map_expr_lang = create_map([lit(x) for x in chain(*map_lang.items())])

(df_filmes
 .withColumn("original_language", when(col("original_language").isin(list(map_lang.keys())), map_expr_lang[col("original_language")]).otherwise(col("original_language"))) # padroniza valores
 .groupBy("original_language")
  .agg(count(lit(1)).alias("qtd_valores"))
  .sort(df_filmes.original_language)).display()

original_language,qtd_valores
null,11
en,19
es,10
pt,21


In [0]:
from pyspark.sql.functions import col, count, lit, create_map, when
from itertools import chain

map_lang      = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
map_expr_lang = create_map([lit(x) for x in chain(*map_lang.items())])

df_filmes = (df_filmes
             .withColumn(
                 "original_language", 
                 when(col("original_language").isin(list(map_lang.keys())), map_expr_lang[col("original_language")]).otherwise(col("original_language"))
             ))

### 1.8 `title`

In [0]:
df_filmes.createOrReplaceTempView('filmes_primeiros_tratamentos')

In [0]:
%sql
select content_type, title, count(*) as total_linhas
from filmes_primeiros_tratamentos
group by 1,2 order by 2,1

content_type,title,total_linhas
miniseries,A Grande Virada,1
series,A Grande Virada 12,1
movie,A Grande Virada 26,1
movie,A Grande Virada 27,1
series,Amor em Lisboa,1
miniseries,Amor em Lisboa 21,1
movie,Amor em Lisboa 29,1
miniseries,Café com Código 55,1
movie,Café com Código 56,1
null,Café com Código 6,1


Ao verificar os valores presentes na coluna `title`, verificamos que em muitas o titulo do filme termina com um número. Mas como se tratam de números muito grandes para representarem sequências de filmes anteriores, vou apenas considerar que se trata de um "erro" e remover esses números.

In [0]:
from pyspark.sql.functions import col, lower, trim, split

(df_filmes
 .withColumn("title", lower(trim(col("title"))))
 .withColumn("title_sn", split(df_filmes["title"], r"\s\d+$")[0])
 .select("title", "title_sn")
 .distinct()).display()

title,title_sn
mar aberto 7,mar aberto
horizonte 2,horizonte
planeta azul 16,planeta azul
dados em fuga 17,dados em fuga
dados em fuga,dados em fuga
código final 22,código final
mar aberto 25,mar aberto
a grande virada 27,a grande virada
horizonte 30,horizonte
noites de verão,noites de verão


In [0]:
from pyspark.sql.functions import col, lower, trim, split

df_filmes = (df_filmes
             .withColumn("title", lower(trim(col("title"))))
             .withColumn("title", split(df_filmes["title"], r"\s\d+$")[0]))

### 1.9 Resume operações aplicadas

In [0]:
from pyspark.sql.functions import col, lower, trim, count, lit, create_map, when, expr, split
from itertools import chain
from datetime import datetime

df_filmes = spark.read.csv(f"{files_path}/catalogo_filmes.csv", header=True, inferSchema=True)
print((df_filmes.count(), len(df_filmes.columns)))

df_filmes = df_filmes.dropDuplicates(subset=['movie_id'])

map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
map_expr_genre = create_map([lit(x) for x in chain(*map_genre.items())])
df_filmes      = (df_filmes
                  .withColumn("genre", lower(trim(col("genre"))))
                  .withColumn("genre", when(col("genre").isin(list(map_genre.keys())), map_expr_genre[col("genre")]).otherwise(col("genre"))))

current_year = int(datetime.now().year)
df_filmes    = (df_filmes
                .withColumn('release_year', expr('try_cast(release_year as int)'))
                .withColumn('release_year', when(col("release_year") <= current_year, col("release_year")).otherwise(None)))

df_filmes = (df_filmes.withColumn('duration_minutes', expr('try_cast(duration_minutes as int)')))

df_filmes = (df_filmes.withColumn("content_type", lower(trim(col("content_type")))))

map_lang      = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
map_expr_lang = create_map([lit(x) for x in chain(*map_lang.items())])
df_filmes     = (df_filmes
                 .withColumn(
                     "original_language", 
                     when(col("original_language").isin(list(map_lang.keys())), map_expr_lang[col("original_language")])
                     .otherwise(col("original_language"))))

df_filmes = (df_filmes
             .withColumn("title", lower(trim(col("title"))))
             .withColumn("title", split(df_filmes["title"], r"\s\d+$")[0]))

df_filmes = df_filmes.fillna("sem informacao") # nao fiz uma sessao para essa transformacao
print((df_filmes.count(), len(df_filmes.columns)))

(62, 7)
(61, 7)


In [0]:
df_filmes.printSchema()

root
 |-- movie_id: string (nullable = false)
 |-- title: string (nullable = false)
 |-- genre: string (nullable = false)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- content_type: string (nullable = false)
 |-- original_language: string (nullable = false)



## 2 Usuários

### 2.1 Lendo arquivo de Usuários

In [0]:
df_usuarios = spark.read.csv(f"{files_path}/usuarios.csv", header=True, inferSchema=True)
print((df_usuarios.count(), len(df_usuarios.columns)))

(82, 6)


In [0]:
df_usuarios.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- subscription_type: string (nullable = true)



In [0]:
df_usuarios.limit(5).display()

user_id,user_name,age,country,signup_date,subscription_type
U0001,Hugo Oliveira,31,Brasil,2024-10-16,basic
U0002,Ana Santos,43,USA,11-23-2024,premium
U0003,Nicolas Souza,unknown,Portugal,08-27-2023,null
U0004,Felipe Pereira,64,Colombia,26/08/2024,Premium
U0005,Diego Santos,37,Argentina,2023-10-12,BASIC


In [0]:
df_usuarios.createOrReplaceTempView('usuarios')

In [0]:
%sql
-- conta quantidade de valores unicos em cada coluna
select 'unicos' as valores
    ,count(distinct user_id) as user_id
    ,count(distinct user_name) as user_name
    ,count(distinct age) as age
    ,count(distinct country) as country
    ,count(distinct signup_date) as signup_date
    ,count(distinct subscription_type) as subscription_type
    ,count(*) as total_linhas
from usuarios
union
select 'nulos' as valores
    ,count_if(user_id is null) as user_id
    ,count_if(user_name is null) as user_name
    ,count_if(age is null) as age
    ,count_if(country is null) as country
    ,count_if(signup_date is null) as signup_date
    ,count_if(subscription_type is null) as subscription_type
    ,count(*) as total_linhas
from usuarios

valores,user_id,user_name,age,country,signup_date,subscription_type,total_linhas
unicos,81,61,42,14,81,7,82
nulos,0,0,2,0,0,9,82


### 2.2 Resolve duplicação em `user_id`

In [0]:
%sql
-- verifica linha ID de filme duplicado
select a.*
from usuarios a 
    inner join (select user_id from usuarios group by user_id having count(*) > 1) b on a.user_id = b.user_id

user_id,user_name,age,country,signup_date,subscription_type
U0010,Carla Oliveira,57,Espanha,17/11/2024,premium
U0010,Carla Oliveira,57,Espanha,17/11/2024,premium


In [0]:
# remove duplicatas considerando coluna user_id
df_usuarios = df_usuarios.dropDuplicates(subset=['user_id'])
print((df_usuarios.count(), len(df_usuarios.columns)))

(81, 6)


In [0]:
df_usuarios.createOrReplaceTempView('usuarios_id_unicos')

### 2.3 Verifica valores de `user_name`

In [0]:
%sql
-- verifica linhas com nome duplicado
select a.*, if(b.user_id is not null, 1, 0) as flag_nome_duplicado
from usuarios_id_unicos a 
    left join (select user_id from usuarios_id_unicos qualify count(*) over (partition by user_name) > 1) b on a.user_id = b.user_id
order by flag_nome_duplicado desc, a.user_name

user_id,user_name,age,country,signup_date,subscription_type,flag_nome_duplicado
U0017,Ana Santos,51,United States,01-26-2024,trial,1
U0002,Ana Santos,43,USA,11-23-2024,premium,1
U0065,Carla Ruescas,-5,BR,2025-04-05,BASIC,1
U0031,Carla Ruescas,31,BR,24/05/2023,basic,1
U0040,Carla Silva,64,Chile,11-18-2024,null,1
U0026,Carla Silva,30,Espanha,2023-01-08,Premium,1
U0033,Felipe Pereira,19,Argentina,12/05/2024,free,1
U0004,Felipe Pereira,64,Colombia,26/08/2024,Premium,1
U0052,Hugo Costa,63,Colombia,12/03/2025,standard,1
U0064,Hugo Costa,21,Spain,06-11-2024,premium,1


Vou considerar que se tratam de pessoas homônimas, por terem idades diferentes e estarem registradas países diferentes. Portanto, nenhum tratamento será aplicado nessa coluna.

### 2.4 Converte coluna `age` para INT e substitui valores inválidos

In [0]:
df_usuarios.select("age").distinct().sort(df_usuarios.age.asc()).display()

age
null
-5
16
18
19
20
21
22
23
24


In [0]:
from pyspark.sql.functions import col, expr, when

df_usuarios = (df_usuarios
               .withColumn('age', expr('try_cast(age as int)'))
               .withColumn('age', when(col("age") > 0, col("age")).otherwise(None)))

In [0]:
df_usuarios.select("age").distinct().sort(df_usuarios.age.asc()).limit(3).display()

age
null
16
18


### 2.5 Padroniza valores de `country`

In [0]:
%sql
-- verifica linhas com nome duplicado
select country, count(*)
from usuarios_id_unicos
group by 1
order by 1

country,count(*)
Argentina,13
BR,6
Brasil,2
Brazil,5
Chile,3
Colombia,7
Colômbia,6
Espanha,4
Mexico,7
México,6


In [0]:
from pyspark.sql.functions import col, when, create_map, lit, lower, trim
from itertools import chain

mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
mapa_expr_country  = create_map([lit(x) for x in chain(*mapeamento_country.items())])
df_usuarios        = (df_usuarios
                      .withColumn("country", lower(trim(col("country"))))
                      .withColumn("country", when(col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[col("country")]).otherwise(col("country"))))


In [0]:
df_usuarios.select("country").distinct().display()

country
usa
mexico
colombia
portugal
argentina
chile
espanha
brasil


### 2.6 Converte coluna `signup_date` para DATE, considerando os diferentes formatos de valores existentes

In [0]:
# cria df apenas com datas
df_dates = df_usuarios.select("signup_date").distinct()
df_dates = df_usuarios.select(col("signup_date").alias("original_value")).distinct()
df_dates.display()

original_value
2024-05-27
31/12/2024
04/02/2024
2023-03-12
11/10/2024
2023-02-09
2023-03-09
06/09/2023
12/05/2024
11-18-2024


In [0]:
from pyspark.sql.functions import col, coalesce, try_to_date, split

# funcao para converter multiplos formatos de data
def to_date_multi_format(coluna, formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")):
    return coalesce(*[try_to_date(coluna, f) for f in formatos_suportados])

# teste da funcao
df_dates = df_dates.withColumn("cast_value", to_date_multi_format(col("original_value")))

df_dates.filter( col("cast_value").isNull() ).display()

original_value,cast_value


Não queria adicionar o padrão `"yyyy/dd/MM"` nos formatos suportados da UDF, mas se não fizer isso, a base vai ficar com uma linha vazia nessa coluna. Então adicionarei esse formato no final da lista.

In [0]:
def to_date_multi_format(coluna, formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")):
    return coalesce(*[try_to_date(coluna, f) for f in formatos_suportados])
df_usuarios = df_usuarios.withColumn("signup_date", to_date_multi_format(split(df_usuarios["signup_date"], ' ')[0]))

### 2.7 Padroniza os valores de `subscription_type`

In [0]:
df_usuarios.select('subscription_type').distinct().display()

subscription_type
standard
null
free
trial
basic
BASIC
Premium
premium


In [0]:
from pyspark.sql.functions import lower, trim

df_usuarios = df_usuarios.withColumn("subscription_type", lower(trim("subscription_type")))

### 2.8 Resume operações aplicadas

In [0]:
from pyspark.sql.functions import col, expr, when, create_map, lit, lower, trim, coalesce, try_to_date, split
from itertools import chain

df_usuarios = spark.read.csv(f"{files_path}/usuarios.csv", header=True, inferSchema=True)
print((df_usuarios.count(), len(df_usuarios.columns)))

df_usuarios = df_usuarios.dropDuplicates(subset=['user_id'])

df_usuarios = (df_usuarios
               .withColumn('age', expr('try_cast(age as int)'))
               .withColumn('age', when(col("age") > 0, col("age")).otherwise(None)))

mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
mapa_expr_country  = create_map([lit(x) for x in chain(*mapeamento_country.items())])
df_usuarios        = (df_usuarios
                      .withColumn("country", lower(trim(col("country"))))
                      .withColumn("country", when(col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[col("country")]).otherwise(col("country"))))

def to_date_multi_format(coluna, formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")):
    return coalesce(*[try_to_date(coluna, f) for f in formatos_suportados])
df_usuarios = df_usuarios.withColumn("signup_date", to_date_multi_format(split(df_usuarios["signup_date"], ' ')[0]))

df_usuarios = df_usuarios.withColumn("subscription_type", lower(trim("subscription_type")))

print((df_usuarios.count(), len(df_usuarios.columns)))

(82, 6)
(81, 6)


In [0]:
df_usuarios.createOrReplaceTempView('usuarios_tratados')

In [0]:
%sql
-- conta quantidade de valores unicos em cada coluna
select 'inicio' as status_base, 'unicos' as valores
    ,count(*) as total_linhas, count(distinct user_id) as user_id, count(distinct user_name) as user_name, count(distinct age) as age
    ,count(distinct country) as country, count(distinct signup_date) as signup_date, count(distinct subscription_type) as subscription_type
from usuarios
union
select 'inicio' as status_base, 'nulos' as valores
    ,count(*) as total_linhas, count_if(user_id is null) as user_id, count_if(user_name is null) as user_name, count_if(age is null) as age
    ,count_if(country is null) as country, count_if(signup_date is null) as signup_date, count_if(subscription_type is null) as subscription_type
from usuarios
union all
select 'fim' as status_base, 'unicos' as valores
    ,count(*) as total_linhas, count(distinct user_id) as user_id, count(distinct user_name) as user_name, count(distinct age) as age
    ,count(distinct country) as country, count(distinct signup_date) as signup_date, count(distinct subscription_type) as subscription_type
from usuarios_tratados
union
select 'fim' as status_base, 'nulos' as valores
    ,count(*) as total_linhas, count_if(user_id is null) as user_id, count_if(user_name is null) as user_name, count_if(age is null) as age
    ,count_if(country is null) as country, count_if(signup_date is null) as signup_date, count_if(subscription_type is null) as subscription_type
from usuarios_tratados

status_base,valores,total_linhas,user_id,user_name,age,country,signup_date,subscription_type
inicio,unicos,82,81,61,42,14,81,7
inicio,nulos,82,0,0,2,0,0,9
fim,unicos,81,81,61,39,8,79,5
fim,nulos,81,0,0,9,0,0,9


In [0]:
df_usuarios.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- user_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- subscription_type: string (nullable = true)

